**Příklad 1:** Vytvořte HOF `compose`, která bude dělat to samé jako operátor `(.)`, tedy bude skládat dvě funkce.

In [1]:
:t (.)

(.) :: forall b c a. (b -> c) -> (a -> b) -> a -> c

In [2]:
--EX01
compose :: (b -> c) -> (a -> b) -> a -> c
-- zde bude vaše definice compose
compose f g x = f (g x)

In [3]:
-- Test řešení:
(negate `compose` sum) [1, 2, 3]
-- musí fungovat stejně jako
(negate . sum) [1, 2, 3]

-6

-6

**Příklad 2:** Zadefinujte rekurzivně výpočet n-tého členu Fibonacciho posloupnosti:
$$
Fib(x) = \begin{cases}
0 & \text{pro } x = 0 \\
1 & \text{pro } x = 1 \\
Fib(x - 1) + Fib(x - 2) & \text{jinak}
\end{cases}
$$

Nejprve si napište typovou signaturu, pak základní případy a pak rekurzivní sebe vyhodnocení:

In [4]:
-- EX02
fib :: (Eq a, Num a) => a -> a
fib 0 = 0
fib 1 = 1
fib n = fib (n - 1) + fib (n - 2)

**Příklad 3:** Napište funkci `intDiv`, která bude realizovat celočíselné dělení pomocí postupného odčítání dělitele od dělence. Funkce bude vracet dvojici `(podíl, zbytek)`.

Pro připomenutí:
```text
11 / 3:         11  (počáteční stav)  
           - 3 = 8  (počet kroků: 1; kontrola: 8 > 3, pokračuju)
           - 3 = 5  (počet kroků: 2; kontrola: 5 > 3, pokračuju)
           - 3 = 2  (počet kroků: 3; kontrola: 2 < 3, končím)
                    => výsledek:  3, zbytek 2
```

<details>
    <summary>Hint:</summary>
    V ukázce výše vidíte, že od hodnoty postupně odečítáte dělitele a přitom si potřebujete někde pamatovat počet kroků. Budete si tedy muset zadefinovat nějakou pomocnou funkci, která si bude v parametrech předávat „aktuální“ hodnotu a počítadlo. Tahle pomocná funkce bude rekurzivní: ve svém těle nejprve provede <i>kontrolu</i> a podle výsledku buď už vrátí (počítadlo kroků, zbytek), nebo sama sebe vyhodnotí se sníženou hodnotou a zvýšeným počítadlem. (Pokud to chcete udělat „pěkně“, definujte pomocnou funkci pomocí klíčového slova <code>where</code>.)
</details>

In [5]:
-- EX03
intDiv :: Integral a => a -> a -> (a, a)
intDiv nom denom = f nom 0
    where
        f current count
            | current < denom = (count, current)
            | otherwise       = f (current - denom) (count + 1)

In [6]:
-- Test řešení:
intDiv 11 3 -- vrátí (3, 2)

(3,2)

**Příklad 4:** Napište funkci `mc`, která vyhodnocuje funkci [McCarthy 91](https://en.wikipedia.org/wiki/McCarthy_91_function):
$$
mc(n)=\begin{cases}
 n - 10 & \mbox{pokud }n > 100\mbox{ } \\
 mc(mc(n+11)) & \mbox{pokud }n \le 100\mbox{ }
\end{cases}
$$
(Tato kouzelná funkce vrací pro každý svůj vstup, který je $\le 100$, hodnotu 91.)

Využijte syntaxi se strážemi!

In [7]:
-- EX04
mc :: Integer -> Integer
mc n 
  | n > 100   = n - 10
  | otherwise = mc (mc (n + 11))

In [8]:
-- Test řešení:
mc 1
mc 23
mc 99
mc 100
mc 110
mc 120

91

91

91

91

100

110

**Příklad 5:** Napište funkci `safeTail`, která vrátí zbytek seznamu po oddělení hlavičky zabalený v typu `Maybe`. Pokud je seznam prázdný *nebo má jen jeden prvek*, výsledkem bude `Nothing`.

In [9]:
-- EX05
safeTail :: [a] -> Maybe [a]
safeTail [] = Nothing
safeTail [x] = Nothing
safeTail (x:xs) = Just xs

In [10]:
-- Test řešení:
safeTail [] -- vypíše Nothing
safeTail [1] -- vypíše Nothing
safeTail [1,2] -- vypíše Just [2]

Nothing

Nothing

Just [2]

**Příklad 6:** Napište funkci `headTail`, která rozdělí seznam na hlavičku a zbytek. Pokud je seznam prázdný, výsledkem bude `Nothing`, jinak bude výsledkem dvojice `(x, xs)`, kde `x` značí hlavičku a `xs` zbytek seznamu. Zde naopak uvažujte, že prázdný seznam je legitimní zbytek seznamu (to taky více odpovídá skutečné sémantice seznamu než chování v předchozím příkladě).

In [11]:
-- EX06
headTail :: [a] -> Maybe (a, [a])
headTail [] = Nothing
headTail [x] = Just (x, [])
headTail (x:xs) = Just (x, xs)

In [12]:
-- Test řešení:
headTail []  -- vypíše Nothing
headTail [1] -- vypíše Just (1, [])
headTail [1,2] -- vypíše Just (1, [2])

Nothing

Just (1,[])

Just (1,[2])

**Příklad 7:** Napište funkci `bstVals`, která do seznamu uloží hodnoty z BST bez **duplicit**.

<details>
    <summary>Hint 1:</summary>
    Budete potřebovat opět nějakou pomocnou funkci (třeba <code>go</code>), která dostane původní uzel, ale k tomu si ještě bude udržovat nějaký „akumulátor“ – seznam, do kterého bude rekurzivně přidávat hodnoty. Pak <code>bstVals node = go node []</code> a všechna magie se odehraje uvnitř <code>go</code>.
</details>
<details>
    <summary>Hint 2:</summary>
    <code>go Empty acc = acc</code>
</details>

In [13]:
-- definice BST a funkce pro jeho vytvoření
data BST k v = Empty | Node (BST k v) k v (BST k v) deriving (Eq, Show)

insertBST :: Ord k => k -> v -> BST k v -> BST k v
insertBST k v Empty = Node Empty k v Empty
insertBST k v (Node l k0 v0 r)
  | k < k0 = Node (insertBST k v l) k0 v0 r
  | k > k0 = Node l k0 v0 (insertBST k v r)
  | otherwise = Node l k v r

bst1 :: BST Int Char
bst1 =
  insertBST 5 'a' $
  insertBST 2 'b' $
  insertBST 9 'c' $
  insertBST 4 'x' $
  insertBST 7 'a' Empty

In [14]:
-- EX07
bstVals :: Eq a => BST k a -> [a]
bstVals node = go node []
    where
        go Empty acc = acc
        go (Node l _ v r) acc = go r (insIfNotExists v (go l acc))
        -- můžete použít následující funkci, která přidá prvek do seznamu, pokud tam už není
        insIfNotExists :: Eq b => b -> [b] -> [b]
        insIfNotExists x xs = if x `elem` xs then xs else x:xs

In [15]:
-- Test řešení:
bstVals bst1  -- vypíše řetězec (seznam), kde budou znaky 'c', 'a', 'x', 'b' (v libovolném pořadí)

"caxb"

**Příklad 8:** Následující výraz vrátí seznam se všemi násobky 3 mezi 1 a 30. **S využitím kompozice funkcí** `(.)` a funkce `length` vytvořte funkci `howManyMultiples3`, která bude vracet počet násobků 3 uvnitř zadaného seznamu.
```haskell
filter (\x -> rem x 3 == 0) [1..30]
```

In [16]:
-- EX08
howManyMultiples3 :: (Integral a) => [a] -> Int 
-- nepřidávejte funkci explicitně žádný parametr
howManyMultiples3 = length . filter (\x -> rem x 3 == 0)

In [17]:
-- Test řešení:
howManyMultiples3 [1..30]  -- vrátí 10

10

**Příklad 9:** Doplňte funkci `getDups`, která „sesbírá“ do jednoho seznamu hodnoty pro stejné klíče a vrátí jen klíče, které se objevily víckrát.

In [18]:
-- EX08
getDups :: Eq a => [(a,b)] -> [(a,[b])]
getDups xs = go xs []
    where
        go [] _ = []
        go ((k,_):rest) seen
            | k `elem` seen = go rest seen
            | otherwise =
                let values = [v | (key,v) <- xs, key == k]
                in ([(k, values) | length values > 1]) ++ go rest (k:seen)

In [19]:
-- Test řešení:
getDups [(1,'a'),(2,'k'),(1,'b'),(3,'z'),(3,'z'),(1,'c'),(4,'@')] == [(1,"abc"),(3,"zz")]

True

**Příklad 10:** S využitím rekurze implementujte funkci `composeAll`, která *skládá* funkce ze seznamu zleva doprava: `composeAll [(+1),(*2),(^2)] x == (((x + 1) * 2) ^ 2)`.

In [20]:
-- EX10
composeAll :: [a -> a] -> (a -> a)

composeAll [] = id
composeAll (f:fs) = composeAll fs . f

In [21]:
-- Test řešení:
composeAll [(+1),(*2),(^2)] 3
-- vypíše 64, tedy hodnotu ((3+1)*2)^2

64